# <a id='toc1_'></a>[Coleta de Logs](#toc0_)

**Sumário**<a id='toc0_'></a>    
- [Coleta de Logs](#toc1_)    
  - [Introdução](#toc1_1_)    
  - [Desenvolvimento](#toc1_2_)    
  - [Conclusão](#toc1_3_)    
  - [Referências](#toc1_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

## <a id='toc1_1_'></a>[Introdução](#toc0_)

Para calcular os tempos de chegada estimados das ondas superficiais de cada um dos terremotos que obtivemos, utilizamos um script da referência [[1](#referencias)], que utiliza um método denominado **GDM52** para prever as velocidades de fase e de grupo para ondas Rayleigh e Love.

## <a id='toc1_2_'></a>[Desenvolvimento](#toc0_)

Dado a latitude e longitude do ponto de otigem de um terremoto e do ponto onde queremos observar, conseguimos estimar o tempo de chegada das ondas superficiais pela seguinte fórmula:

$$ t = \frac{d}{U}$$

onde $t$ é o tempo de chegada, $d$ é a menor distância pela superfície do planeta e $U$ é a velocidade do grupo.

A distância conseguimos pelo método `gps2dist_azimuth()` do módulo **obspy.geodetics**. Para a velocidade do grupo, iremos utilizar um script em fortran que prevê a velocidade de diferentes sequências de um terremoto com uma origem determinada a partir do modelo `GDM52` [[1](#referencias)]. Para isso, precisamos de um compilador de Fortran. Nesse caso, utilizaremos o do Octave que já vem instalado nos computadores da Ilum.

A função para executar o script fica assim:

In [6]:
import subprocess
from pathlib import Path
import pandas as pd
import os

def calculate_GDM52_dispersion(
    model="Love",
    dispersion_type="path",
    origin_point=(0, 0),
    receiver_point=(0, 0)
):
    """
    Calculate surface-wave dispersion curves using the GDM52 model.

    Runs the GDM52 Fortran program to obtain phase and group velocities
    for Love or Rayleigh waves, either at a single geographic point or
    along a path between two geographic coordinates.

    Parameters
    ----------
    model : {"Love", "Rayleigh"}, default="Love"
        Surface-wave type to calculate. "Love" selects Love waves,
        while "Rayleigh" switches the GDM52 program to Rayleigh waves.

    dispersion_type : {"point", "path"}, default="path"
        Type of dispersion calculation:
        - "point": calculates dispersion at a single geographic point,
          specified by `origin_point`.
        - "path": calculates dispersion along the path between
          `origin_point` and `receiver_point`.

    origin_point : tuple of (float, float), default=(0, 0)
        Geographic coordinates of the calculation point or path origin,
        expressed as (latitude, longitude), in degrees.

    receiver_point : tuple of (float, float), default=(0, 0)
        Geographic coordinates of the path endpoint, expressed as
        (latitude, longitude), in degrees. Used only when
        `dispersion_type="path"`.

    Returns
    -------
    pandas.DataFrame
        Dispersion results with the following columns:
        - f(mHz): Frequency in millihertz.
        - c(PREM): Reference phase velocity from PREM.
        - c: Phase velocity predicted by GDM52.
        - dc(%): Percentage difference between GDM52 and PREM phase
          velocities.
        - U(PREM): Reference group velocity from PREM.
        - U: Group velocity predicted by GDM52.
        - dU(%): Percentage difference between GDM52 and PREM group
          velocities.

    Raises
    ------
    FileNotFoundError
        If the GDM52 output file cannot be found.
    ValueError
        If the output file contains rows that cannot be parsed as
        floating-point values.
    RuntimeError
        If the GDM52 program reports an error or fails to execute
        successfully.

    Notes
    -----
    This function requires the compiled `GDM52_dispersion.exe` executable
    and the GDM52 program files to be present in the expected directory.

    The function uses the program's generated output file,
    `GDM52_dispersion.out`, to construct the returned DataFrame.

    For path calculations, the resulting dispersion curve can be used
    to investigate frequency-dependent surface-wave propagation between
    the specified locations.

    """
    input_text = ""
    
    if model == "Rayleigh":
        input_text += f"77\n"
        
    if dispersion_type == "point":
        input_text += f"2\n"
        input_text += f"{origin_point[0]} {origin_point[1]}\n"
    elif dispersion_type == "path":
        input_text += f"6\n"
        input_text += f"{origin_point[0]} {origin_point[1]}\n"
        input_text += f"{receiver_point[0]} {receiver_point[1]}\n"
    
    input_text += f"99"
    
    directory = Path.cwd() / "GDM52_program"

    exe = directory / "GDM52_dispersion.exe"
    output = directory / "GDM52_dispersion.out"

    env = os.environ.copy()
    octave_bin = r"C:\Program Files\GNU Octave\Octave-10.2.0\mingw64\bin"

    env["PATH"] = octave_bin + os.pathsep + env.get("PATH", "")

    result = subprocess.run(
        [str(exe)],
        input="99\n",
        capture_output=True,
        text=True,
        cwd=str(directory),
        env=env,
        check=False,
    )
    
    if result.stderr:
        print("ERRO -> ", result.stderr)
        raise
    
    with open(output) as f:
        out = f.read().split(f"\n")
        out_cols = ['f(mHz)', 'c(PREM)', 'c', 'dc(%)', 'U(PREM)', 'U', 'dU(%)']
        out_rows = [[float(x) for x in s.strip().split()] for s in out[5:-1]]
        
        out_df = pd.DataFrame(columns=out_cols, data=out_rows)
        
    return out_df

Vamos testar o tempo de chegada das ondas Rayleighs partindo de uma determinada latitude e longitude até o Sirius.

In [7]:
SIRIUS_LOCATION = (-22.807606, -47.052526)

result = calculate_GDM52_dispersion(
    model="Rayleigh",
    dispersion_type="path",
    origin_point=(7.19, -81.32),
    receiver_point=SIRIUS_LOCATION
)

print(result)

    f(mHz)  c(PREM)      c  dc(%)  U(PREM)      U  dU(%)
0    4.000    5.071  5.099  0.550    4.386  4.396  0.231
1    4.364    5.007  5.033  0.519    4.382  4.392  0.219
2    4.727    4.954  4.978  0.493    4.381  4.392  0.244
3    5.091    4.909  4.932  0.476    4.382  4.396  0.318
4    5.455    4.871  4.894  0.469    4.382  4.402  0.446
..     ...      ...    ...    ...      ...    ...    ...
95  38.545    4.260  3.996 -6.203    3.730  3.492 -6.371
96  38.909    4.254  3.990 -6.203    3.718  3.490 -6.133
97  39.273    4.248  3.985 -6.202    3.706  3.485 -5.964
98  39.636    4.243  3.980 -6.199    3.693  3.477 -5.868
99  40.000    4.237  3.974 -6.195    3.682  3.466 -5.847

[100 rows x 7 columns]


Agora, basta calcularmos a distância para encontrarmos os tempos de chegada de cada onda. Vamos montar uma função que faz exatamente isso:

In [8]:
from obspy.geodetics import gps2dist_azimuth

def calculate_arrival_times(origin_time, origin_point, receiver_point):
    rayleigh = calculate_GDM52_dispersion(
        model="Rayleigh",
        dispersion_type="path",
        origin_point=origin_point,
        receiver_point=receiver_point
    )
    
    rayleigh["type"] = "rayleigh"
    
    love = calculate_GDM52_dispersion(
        model="Love",
        dispersion_type="path",
        origin_point=origin_point,
        receiver_point=receiver_point
    )
    
    love["type"] = "love"
    
    dist_m, az, baz = gps2dist_azimuth(origin_point[0], origin_point[1], receiver_point[0], receiver_point[1])
    
    df = pd.concat([rayleigh, love]).reset_index(drop=True)
    
    df = df.reindex(["f(mHz)", "T(s)", "U", "type", "arrival_time"], axis=1)
    
    df["T(s)"] = 1000 / df["f(mHz)"]
    df["travel_time_s"] = (dist_m / 1000) / df["U"]
    df["arrival_time"] = (pd.Timestamp(origin_time) + pd.to_timedelta(df["travel_time_s"], unit="s")).dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    
    return df

E agora, vamos testar com aquele mesmo exemplo

In [13]:
df = calculate_arrival_times(
    origin_time="2026-10-09T17:56:07Z",
    origin_point=(7.19, -81.32),
    receiver_point=SIRIUS_LOCATION
)

df

,f(mHz),T(s),U,type,arrival_time,travel_time_s
0,4.000,250.000000,4.396,rayleigh,2026-10-09T18:15:03Z,1136.045709
1,4.364,229.147571,4.392,rayleigh,2026-10-09T18:15:04Z,1137.080359
2,4.727,211.550666,4.392,rayleigh,2026-10-09T18:15:04Z,1137.080359
3,5.091,196.425064,4.396,rayleigh,2026-10-09T18:15:03Z,1136.045709
4,5.455,183.318057,4.402,rayleigh,2026-10-09T18:15:01Z,1134.497259
...,...,...,...,...,...,...
195,38.545,25.943702,3.492,love,2026-10-09T18:19:57Z,1430.142307
196,38.909,25.700995,3.490,love,2026-10-09T18:19:57Z,1430.961872
197,39.273,25.462786,3.485,love,2026-10-09T18:20:00Z,1433.014902
198,39.636,25.229589,3.477,love,2026-10-09T18:20:03Z,1436.312032


Agora temos cada onda e o tempo de chegada estimado!

## <a id='toc1_3_'></a>[Conclusão](#toc0_)

Com o modelo **GDM52**, portanto, conseguimos estimar a velocidade de cada grupo de ondas pela superfície, o que pode ser utilizado para calcular o tempo de chegada esperado de cada onda a partir da sua localização de origem.

O próximo passo é entendermos como calcular as ondas de corpo, e como relacionar os dados obtidos nos sismógrafos com os tempos previstos.

<span id="referencias"></span>
## <a id='toc1_4_'></a>[Referências](#toc0_)

1. EKSTRÖM, Göran. *GDM52: Seismic data processing*. [S. l.]: Columbia University, [s. d.]. Disponível em: https://www.ldeo.columbia.edu/~ekstrom/Projects/SWP/GDM52.html. Acesso em: 6 nov. 2026.